# Supplementary notebook 1. From link copying to strategy copying (Fig. S2)

Reproduces SI Fig. S2. Reads no data. Run time about 2 minutes. `QUICK = True` in the first code cell shortens the runs for a test.

In [1]:
QUICK = False
T_RUN = 3000      # the run read by Fig. 4 and the copying analysis of Fig. S2
T_POP = 5000      # the four conditions, the hypothesis tests, the sweeps and Fig. 5; the same seeds,
                  # so the first 3,000 steps of these runs are the runs of Fig. 4

import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from math import log2

plt.rcParams.update({'figure.dpi': 110, 'font.size': 8, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.linewidth': 0.7,
                     'xtick.major.width': 0.7, 'ytick.major.width': 0.7,
                     'legend.frameon': False, 'font.family': 'DejaVu Sans'})

CEIL = log2(100)          # 6.6439 bits, the even-coupling ceiling at N = 100
BA_KAPPA = 10.10          # matched scale-free ensemble, control-entropy paper Table 1
BA_DH_DEG = 0.357         # its degree-weighted compression
BA_RDEG = -0.17           # its degree assortativity
N = 100
COL = {'baseline': '#7A7A7A', 'explore': '#2E7D4E', 'imitate': '#2E5090', 'both': '#A4161A'}
CONDS = {'baseline': dict(pr=0.0, pe=0.0), 'explore': dict(pr=0.0, pe=0.05),
         'imitate': dict(pr=0.3, pe=0.0), 'both': dict(pr=0.3, pe=0.05)}

def seeds(n):
    return list(range(n if not QUICK else max(2, n // 5)))

print('ceiling log2 N =', round(CEIL, 4), 'bits')

ceiling log2 N = 6.6439 bits


## Step 0. The model

In [2]:
import numpy as np
import networkx as nx

LOG2 = np.log(2.0)


# ----------------------------------------------------------------------
# measures (definitions of the second study)
# ----------------------------------------------------------------------
def control_entropy(v):
    """Control entropy in bits of a non-negative vector of shares (Eq. 11)."""
    v = np.asarray(v, float)
    tot = v.sum()
    if tot <= 0:
        return 0.0
    p = v / tot
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())


def eigen_shares(W):
    """Shares from the leading eigenvector of a symmetric coupling matrix."""
    vals, vecs = np.linalg.eigh(W)
    v = np.abs(vecs[:, -1])
    return v / v.sum()


def adjacency_measures(A, W):
    """Measures read from the unweighted structure, computed as in Table 1 of the
    control-entropy paper: eigenvector weighting on the largest connected component,
    degree assortativity on the whole graph."""
    G = nx.from_numpy_array(A.astype(int))
    deg = A.sum(axis=1)
    s = W.sum(axis=1)
    iu, ju = np.nonzero(np.triu(A, 1))

    def edge_corr(v):
        x, y = np.concatenate([v[iu], v[ju]]), np.concatenate([v[ju], v[iu]])
        return float(np.corrcoef(x, y)[0, 1]) if x.std() > 1e-12 else float('nan')

    r_deg, r_str = edge_corr(deg.astype(float)), edge_corr(s)
    if not nx.is_connected(G):
        G = G.subgraph(max(nx.connected_components(G), key=len)).copy()
    c = nx.eigenvector_centrality_numpy(G)
    v = np.abs(np.array([c[n] for n in G.nodes()]))
    return dict(H_adj=control_entropy(v), r_deg=float(r_deg), r_str=r_str, gc=G.number_of_nodes())


# ----------------------------------------------------------------------
# the five rules, one function each
# ----------------------------------------------------------------------
def ring_lattice(N, k):
    """Regular ring lattice: every element coupled to its k nearest neighbours."""
    return nx.to_numpy_array(nx.watts_strogatz_graph(N, k, 0.0), dtype=bool)


def softmax_allocation(A, beta, shat):
    """Eq 1. Element i splits one unit of coupling over its partners j with weights
    exp(beta_i * shat_j). beta_i = 0 gives even coupling."""
    L = np.where(A, beta[:, None] * shat[None, :], -np.inf)
    L = L - L.max(axis=1, keepdims=True)
    E = np.where(A, np.exp(L), 0.0)
    return E / E.sum(axis=1, keepdims=True)


def commitment(phi0, phi1, C, fm):
    """Eq 2. Commitment of an element, in the linear form of the published model,
    f = phi0 + phi1 * C, truncated to [0, 1 - fm]."""
    return np.clip(phi0 + phi1 * C, 0.0, 1.0 - fm)


def protection(f, c, pmax, chalf):
    """Eq 8. Saturating protection bought by committed capability f * c."""
    x = f * c
    return np.where(x > 0, pmax / (1.0 + chalf / np.maximum(x, 1e-12)), 0.0)


def fermi(dc, s):
    """Eq 10. Probability of copying a partner whose capability is higher by dc."""
    return 1.0 / (1.0 + np.exp(-s * dc))


def logistic(z, g, theta):
    """Response function of an element, used inside the spread (Eq 5)."""
    return 1.0 / (1.0 + np.exp(-g * (z - theta)))


class ElementModel:
    """Network-agent model of element-level exploration and imitation.

    Each element holds
      beta_i        direction of coupling (0 = even coupling, > 0 = toward stronger partners)
      phi0_i, phi1_i commitment strategy, giving f_i = phi0_i + phi1_i * C_i
      c_i           capability, the accumulated stimulation it compares with partners
    Each coupling (i, j) holds a weight w_ij.
    """

    def __init__(self, N=100, k=6, seed=0,
                 pr=0.3, pe=0.05, sigma=0.1, sigma_f=0.01, s=2.0,   # strategy dynamics
                 M=10, rho=0.5, nE=5, switch=None, psi=0.3,        # feedback, two channels
                 gamma=1.0, pn=0.02, pl=0.3, lam=1.0, g=4.0, theta=1.0,   # seeding and spread
                 fm=0.1, pmax=1.0, chalf=0.1, f0=0.1,              # capability and protection
                 eta=0.05, delta=0.01, w0=1.0,                     # consolidation
                 pw=2.0, kmin=2, loss=1.0,                         # pruning and its cost
                 spread=True, two_part=True):
        self.p = dict(N=N, k=k, pr=pr, pe=pe, sigma=sigma, sigma_f=sigma_f, s=s, M=M, rho=rho,
                      nE=nE, switch=switch, psi=psi, gamma=gamma, pn=pn, pl=pl, lam=lam, g=g,
                      theta=theta, fm=fm, pmax=pmax, chalf=chalf, f0=f0, eta=eta, delta=delta,
                      w0=w0, pw=pw, kmin=kmin, loss=loss, spread=spread, two_part=two_part)
        self.rng = np.random.default_rng(seed)
        self.N, self.k = N, k
        self.A = ring_lattice(N, k)
        self.W = self.A.astype(float) * w0
        self.beta = np.zeros(N)
        self.phi0 = np.full(N, f0)
        self.phi1 = np.zeros(N)
        self.c = np.ones(N)
        self.x = np.zeros(N)          # activation of the previous step
        self.v = np.ones(N) / np.sqrt(N)   # leading eigenvector, kept warm
        self.t = 0
        self.n_act = 0.0
        self.n_rounds = 0.0
        self.set_essential(self.rng.choice(N, nE, replace=False))

    # ---------- state helpers ----------
    def set_essential(self, E):
        self.E = np.asarray(E)
        self.isE = np.zeros(self.N, bool)
        self.isE[self.E] = True

    def strengths(self):
        return self.W.sum(axis=1)

    def shat(self):
        s = self.strengths()
        return s / s.max()

    def centrality(self, iters=8):
        """Leading eigenvector of W by power iteration, warm started, scaled to a
        maximum of one. This is the C of the commitment rule (Eq 2)."""
        v = self.v
        for _ in range(iters):
            v = self.W @ v
            n = np.linalg.norm(v)
            if n < 1e-12:
                v = np.ones(self.N) / np.sqrt(self.N)
                break
            v = v / n
        self.v = np.abs(v)
        return self.v / max(self.v.max(), 1e-12)

    def allocation(self):
        return softmax_allocation(self.A, self.beta, self.shat())

    # ---------- one step ----------
    def feedback(self):
        """Eq 3. Two channels of information.
        Process feedback is intrinsic: every element receives afferent input in
        proportion to its own activation in the previous step.
        Result feedback is augmented: M packets, each landing on a task-essential
        element with probability rho and on any element otherwise."""
        p, rng = self.p, self.rng
        u = p['psi'] * self.x.copy()
        if p['M'] > 0:
            to_E = rng.random(p['M']) < p['rho']
            tgt = np.where(to_E, rng.choice(self.E, p['M']), rng.integers(0, self.N, p['M']))
            u = u + np.bincount(tgt, minlength=self.N).astype(float)
        return u

    def spread(self, u, a):
        """Eq 4 and Eq 5. Stimulus is seeded where information arrives and then
        travels along the couplings.
        Seeding: element i is stimulated with probability 1 - (1 - p_n) exp(-gamma u_i),
        so an element fires spontaneously at rate p_n and more often when information
        reaches it.
        Spread: an element already stimulated passes activation to partner j with
        the coupling it allocates, and j responds with probability
        pl * logistic(g (d_j - theta)). Each element activates at most once per step.
        The cascade is run to completion inside the step."""
        p, rng, N = self.p, self.rng, self.N
        act = rng.random(N) < (1.0 - (1.0 - p['pn']) * np.exp(-p['gamma'] * u))   # Eq 4
        if not p['spread']:
            self.n_act, self.n_rounds = float(act.sum()), 0.0
            return act
        frontier = act.copy()
        rounds = 0
        while frontier.any() and rounds < 20:
            # drive reaching each candidate through the allocation of active elements
            d = p['lam'] * self.k * (frontier @ a)
            prob = p['pl'] * logistic(d, p['g'], p['theta'])       # Eq 5
            new = (rng.random(N) < prob) & ~act
            act |= new
            frontier = new
            rounds += 1
        self.n_act, self.n_rounds = float(act.sum()), float(rounds)
        return act

    def step(self):
        p, rng, N = self.p, self.rng, self.N
        if p['switch'] and self.t > 0 and self.t % p['switch'] == 0:
            self.set_essential(rng.choice(N, p['nE'], replace=False))

        C = self.centrality()
        a = self.allocation()                                     # Eq 1
        f = commitment(self.phi0, self.phi1, C, p['fm']) if p['two_part'] else np.zeros(N)  # Eq 2
        u = self.feedback()                                       # Eq 3
        act = self.spread(u, a)                                   # Eq 4, Eq 5
        x = act.astype(float)
        self.x = x

        # Eq 6. Payoff is the stimulation the element gathers. It reaches partners
        # through its own allocation and it integrates in proportion to the coupling
        # it holds, so both the direction and the amount of coupling enter.
        s = self.strengths()
        pay = (s / s.mean()) * (a @ x)
        self.c = pay + np.clip(1.0 - p['fm'] - f, 0.0, None) * self.c   # Eq 7

        # Eq 8 and Eq 9. Commitment protects a coupling against decay.
        P = protection(f, self.c, p['pmax'], p['chalf'])
        Pij = 0.5 * (P[:, None] + P[None, :])
        abar = 0.5 * (a + a.T)
        self.W = np.where(self.A,
                          p['w0'] + (1 - p['delta'] * (1 - Pij)) * (self.W - p['w0'])
                          + p['eta'] * self.k * abar * np.outer(x, x), 0.0)

        # Eq S1. Long-term change: a coupling falls out of use, and unless the
        # capability committed to it holds it, it is lost and replaced elsewhere.
        n_rw = int(p['pw']) + (1 if rng.random() < p['pw'] - int(p['pw']) else 0)
        for _ in range(n_rw):
            self._rewire(P)

        # Eq 10. Imitation along a coupling, synchronous, two independent passes.
        b_old, f0_old, f1_old, c_old = self.beta.copy(), self.phi0.copy(), self.phi1.copy(), self.c.copy()
        for i in np.flatnonzero(rng.random(N) < p['pr']):
            nb = np.flatnonzero(self.A[i])
            j = nb[rng.integers(nb.size)]
            if rng.random() < fermi(c_old[j] - c_old[i], p['s']):
                self.beta[i] = b_old[j]
        if p['two_part']:
            for i in np.flatnonzero(rng.random(N) < p['pr']):
                nb = np.flatnonzero(self.A[i])
                j = nb[rng.integers(nb.size)]
                if rng.random() < fermi(c_old[j] - c_old[i], p['s']):
                    self.phi0[i], self.phi1[i] = f0_old[j], f1_old[j]

        # Eq S2. Exploration of both parts of the strategy.
        ex = rng.random(N) < p['pe']
        self.beta[ex] += rng.normal(0.0, p['sigma'], ex.sum())
        if p['two_part']:
            ex0 = rng.random(N) < p['pe']
            self.phi0[ex0] += rng.normal(0.0, p['sigma_f'], ex0.sum())
            ex1 = rng.random(N) < p['pe']
            self.phi1[ex1] += rng.normal(0.0, p['sigma_f'], ex1.sum())
        self.t += 1
        return x

    def _rewire(self, P):
        """One element drops its weakest coupling, unless protection holds it, and
        attaches to a new partner chosen with exp(beta_i * shat_j)."""
        p, rng = self.p, self.rng
        i = rng.integers(self.N)
        deg = self.A.sum(axis=1)
        nb = np.flatnonzero(self.A[i])
        nb = nb[deg[nb] > p['kmin']]
        if deg[i] <= p['kmin'] or nb.size == 0:
            return
        j = nb[np.argmin(self.W[i, nb])]
        if rng.random() < 0.5 * (P[i] + P[j]):     # the coupling is held
            return
        # the route is lost, and with it the capability the two elements had built
        # through it. With loss = 1 this is the capability loss of the published model.
        self.c[i] *= (1.0 - p['loss'])
        self.c[j] *= (1.0 - p['loss'])
        cand = np.flatnonzero(~self.A[i])
        cand = cand[cand != i]
        if cand.size == 0:
            return
        z = self.beta[i] * self.shat()[cand]
        w = np.exp(z - z.max())
        new = rng.choice(cand, p=w / w.sum())
        self.A[i, j] = self.A[j, i] = False
        self.W[i, j] = self.W[j, i] = 0.0
        self.A[i, new] = self.A[new, i] = True
        self.W[i, new] = self.W[new, i] = p['w0']

    # ---------- measures ----------
    def measures(self, adj=False):
        s = self.strengths()
        deg = self.A.sum(axis=1)
        top = np.argsort(s)[-len(self.E):]
        nbE = self.A[self.E].any(axis=0) & ~self.isE
        tot = s.sum()
        C = self.centrality()
        f = commitment(self.phi0, self.phi1, C, self.p['fm'])
        return dict(H_str=control_entropy(s), H_eig=control_entropy(eigen_shares(self.W)),
                    H_deg=control_entropy(deg), kappa=(deg ** 2).mean() / deg.mean(),
                    beta_mean=self.beta.mean(), beta_var=self.beta.var(),
                    f_mean=float(f.mean()), phi0_mean=float(self.phi0.mean()),
                    phi1_mean=float(self.phi1.mean()), c_mean=float(self.c.mean()),
                    act=self.n_act / self.N, rounds=self.n_rounds,
                    hub_align=self.isE[top].mean(),
                    share_E=s[self.isE].sum() / tot, share_nbE=s[nbE].sum() / tot,
                    **(adjacency_measures(self.A, self.W) if adj else {}))

    def run(self, T, every=25, adj=False):
        rec = {}
        for step in range(T):
            self.step()
            if step % every == 0 or step == T - 1:
                m = self.measures(adj=adj)
                m['t'] = self.t
                for key, v in m.items():
                    rec.setdefault(key, []).append(v)
        return {key: np.array(v) for key, v in rec.items()}


def replicate(T, seeds, adj=True, **kw):
    """Run the model for several seeds and return the final measures."""
    out = []
    for sd in seeds:
        m = ElementModel(seed=sd, **kw)
        for _ in range(T):
            m.step()
        out.append(m.measures(adj=adj))
    return {key: np.array([o[key] for o in out]) for key in out[0]}

In [3]:
from scipy.stats import mannwhitneyu

def cliffs_delta(a, b):
    a, b = np.asarray(a), np.asarray(b)
    gt = sum((x > y) for x in a for y in b)
    lt = sum((x < y) for x in a for y in b)
    return (gt - lt) / (len(a) * len(b))

def boot_median(a, reps=5000, seed=0):
    rng2 = np.random.default_rng(seed)
    a = np.asarray(a)
    d = [np.median(rng2.choice(a, len(a))) for _ in range(reps)]
    return np.percentile(d, [2.5, 97.5])

def boot_diff(a, b, reps=5000, seed=0):
    rng2 = np.random.default_rng(seed)
    a, b = np.asarray(a), np.asarray(b)
    d = [np.median(rng2.choice(a, len(a))) - np.median(rng2.choice(b, len(b))) for _ in range(reps)]
    return np.percentile(d, [2.5, 97.5])

def summary(tag, v, fmt='.3f'):
    v = np.asarray(v, float); lo, hi = boot_median(v)
    print(f'{tag}: median {np.median(v):{fmt}} [{lo:{fmt}}, {hi:{fmt}}], '
          f'mean {v.mean():{fmt}}, sd {v.std(ddof=1):{fmt}}, range {v.min():{fmt}} to {v.max():{fmt}}')

def compare(tag, a, b, unit=''):
    u, p = mannwhitneyu(a, b, alternative='two-sided')
    d = cliffs_delta(a, b)
    lo, hi = boot_diff(a, b)
    diff = np.median(a) - np.median(b)
    print(f'{tag}: U = {u:.1f}, p = {p:.4f}, Cliff delta = {d:+.2f}, '
          f'median difference = {diff:+.3f}{unit} [{lo:+.3f}, {hi:+.3f}]')
    return dict(U=u, p=p, d=d, diff=diff, lo=lo, hi=hi)

def star(p):
    return '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'n.s.'

## Step 1. The copying model and its equation

In [4]:
def grow_copying(n, m, p, variant, seed):
    # variant 'link': copy the end of a random link. variant 'partner': copy a partner of a random element.
    rng = np.random.default_rng(seed)
    adj = [[] for _ in range(n)]; ends = []
    for i in range(m + 1):
        for j in range(i + 1, m + 1):
            adj[i].append(j); adj[j].append(i); ends += [i, j]
    deg = np.zeros(n, int); deg[:m + 1] = m
    hist = np.zeros(n); hist[m] = m + 1
    obs = np.zeros(n); den = np.zeros(n); kk = np.arange(n)
    for new in range(m + 1, n):
        two_L = len(ends); chosen = []
        while len(chosen) < m:
            if rng.random() < p:
                t = int(rng.integers(new))
            elif variant == 'link':
                t = ends[rng.integers(two_L)]
            else:
                u = int(rng.integers(new)); t = adj[u][rng.integers(len(adj[u]))]
            if t in chosen:
                continue
            chosen.append(t)
            den += hist; obs[deg[t]] += two_L        # observed 2L Pi(k), summed over draws
        for t in chosen:
            hist[deg[t]] -= 1; deg[t] += 1; hist[deg[t]] += 1
            adj[new].append(t); adj[t].append(new); ends += [new, t]
        deg[new] = m; hist[m] += 1
    return deg, obs, den

def gamma_shifted(deg, kmin, shift):
    # continuous maximum-likelihood exponent of (k + shift)^-gamma above kmin
    k = deg[deg >= kmin]
    return 1 + len(k) / np.log((k + shift) / (kmin + shift - 0.5)).sum()

N_GROW = 4000 if QUICK else 20000
M_GROW = 3
KER = {}
for p, var in [(0.0, 'link'), (0.5, 'link'), (0.0, 'partner')]:
    deg, obs, den = grow_copying(N_GROW, M_GROW, p, var, seed=1)
    k = np.flatnonzero(den > 1e4 if QUICK else den > 1e5)
    est = obs[k] / den[k]
    slope, icpt = np.polyfit(k, est, 1)
    KER[(p, var)] = (k, est)
    print(f'copying model, p = {p}, copy by {var}: 2L Pi(k) = {icpt:.2f} + {slope:.3f} k '
          f'over k = {k.min()} to {k.max()}; the equation gives {p * 2 * M_GROW:.2f} + {1 - p:.3f} k')

DEGS = {}
for p in (0.0, 0.25, 0.5):
    shift = 2 * M_GROW * p / (1 - p)
    est = [gamma_shifted(grow_copying(N_GROW, M_GROW, p, 'link', seed=sd)[0], 10, shift) for sd in range(3)]
    DEGS[p] = grow_copying(N_GROW, M_GROW, p, 'link', seed=1)[0]
    print(f'degree exponent at p = {p}: estimated {np.mean(est):.2f} (three growths), '
          f'mean-field (3 - p)/(1 - p) = {(3 - p) / (1 - p):.2f}')

copying model, p = 0.0, copy by link: 2L Pi(k) = 0.29 + 0.995 k over k = 3 to 49; the equation gives 0.00 + 1.000 k
copying model, p = 0.5, copy by link: 2L Pi(k) = 3.55 + 0.467 k over k = 3 to 45; the equation gives 3.00 + 0.500 k
copying model, p = 0.0, copy by partner: 2L Pi(k) = -4.15 + 1.704 k over k = 3 to 52; the equation gives 0.00 + 1.000 k
degree exponent at p = 0.0: estimated 2.84 (three growths), mean-field (3 - p)/(1 - p) = 3.00
degree exponent at p = 0.25: estimated 3.45 (three growths), mean-field (3 - p)/(1 - p) = 3.67
degree exponent at p = 0.5: estimated 4.61 (three growths), mean-field (3 - p)/(1 - p) = 5.00


## Step 2. The Fermi rule holds the copying step as its neutral case

In [5]:
dc = np.linspace(-6, 6, 241)
print('Fermi rule at s = 0, smallest and largest value over the differences:',
      fermi(dc, 0.0).min(), fermi(dc, 0.0).max())
print('Fermi rule at s = 2: copy probability at a difference of -1, 0, +1:',
      [round(float(fermi(x, 2.0)), 3) for x in (-1.0, 0.0, 1.0)])

Fermi rule at s = 0, smallest and largest value over the differences: 0.5 0.5
Fermi rule at s = 2: copy probability at a difference of -1, 0, +1: [0.119, 0.5, 0.881]


## Step 3. The full model with two logs

In [6]:
KMAX = 101
NWIN = 12          # windows of 250 steps over 3,000

class Logged(ElementModel):
    def __init__(self, *a, **kw):
        super().__init__(*a, **kw)
        self.meet = np.zeros((NWIN, 2)); self.cp_str = np.zeros((NWIN, 2)); self.cp_cap = np.zeros((NWIN, 2))
        self.n_draw = np.zeros(NWIN); self.n_copy = np.zeros(NWIN)
        self.expo = np.zeros((NWIN, KMAX)); self.gain = np.zeros((NWIN, KMAX)); self.loss = np.zeros((NWIN, KMAX))
        self.traj = []

    def win(self):
        return min(NWIN - 1, int(self.t * NWIN / T_RUN))

    def step(self):
        p, rng, N = self.p, self.rng, self.N
        w = self.win()
        self.expo[w] += np.bincount(self.A.sum(axis=1), minlength=KMAX)[:KMAX]
        if p['switch'] and self.t > 0 and self.t % p['switch'] == 0:
            self.set_essential(rng.choice(N, p['nE'], replace=False))
        C = self.centrality()
        a = self.allocation()
        f = commitment(self.phi0, self.phi1, C, p['fm']) if p['two_part'] else np.zeros(N)
        u = self.feedback()
        act = self.spread(u, a)
        x = act.astype(float)
        self.x = x
        s = self.strengths()
        pay = (s / s.mean()) * (a @ x)
        self.c = pay + np.clip(1.0 - p['fm'] - f, 0.0, None) * self.c
        P = protection(f, self.c, p['pmax'], p['chalf'])
        Pij = 0.5 * (P[:, None] + P[None, :])
        abar = 0.5 * (a + a.T)
        self.W = np.where(self.A,
                          p['w0'] + (1 - p['delta'] * (1 - Pij)) * (self.W - p['w0'])
                          + p['eta'] * self.k * abar * np.outer(x, x), 0.0)
        n_rw = int(p['pw']) + (1 if rng.random() < p['pw'] - int(p['pw']) else 0)
        for _ in range(n_rw):
            self._rewire(P)
        b_old, f0_old, f1_old, c_old = self.beta.copy(), self.phi0.copy(), self.phi1.copy(), self.c.copy()
        deg, st = self.A.sum(axis=1), self.strengths()
        for i in np.flatnonzero(rng.random(N) < p['pr']):
            nb = np.flatnonzero(self.A[i])
            j = nb[rng.integers(nb.size)]
            copied = rng.random() < fermi(c_old[j] - c_old[i], p['s'])
            self.n_draw[w] += 1
            if deg[j] != deg[i]: self.meet[w, int(deg[j] > deg[i])] += 1
            if copied:
                self.n_copy[w] += 1
                if st[j] != st[i]: self.cp_str[w, int(st[j] > st[i])] += 1
                if c_old[j] != c_old[i]: self.cp_cap[w, int(c_old[j] > c_old[i])] += 1
                self.beta[i] = b_old[j]
        if p['two_part']:
            for i in np.flatnonzero(rng.random(N) < p['pr']):
                nb = np.flatnonzero(self.A[i])
                j = nb[rng.integers(nb.size)]
                if rng.random() < fermi(c_old[j] - c_old[i], p['s']):
                    self.phi0[i], self.phi1[i] = f0_old[j], f1_old[j]
        ex = rng.random(N) < p['pe']
        self.beta[ex] += rng.normal(0.0, p['sigma'], ex.sum())
        if p['two_part']:
            ex0 = rng.random(N) < p['pe']
            self.phi0[ex0] += rng.normal(0.0, p['sigma_f'], ex0.sum())
            ex1 = rng.random(N) < p['pe']
            self.phi1[ex1] += rng.normal(0.0, p['sigma_f'], ex1.sum())
        self.t += 1
        if self.t % 25 == 0:
            self.traj.append((self.t, self.beta.mean(), self.shat().mean()))
        return x

    def _rewire(self, P):
        p, rng = self.p, self.rng
        i = rng.integers(self.N)
        deg = self.A.sum(axis=1)
        nb = np.flatnonzero(self.A[i])
        nb = nb[deg[nb] > p['kmin']]
        if deg[i] <= p['kmin'] or nb.size == 0:
            return
        j = nb[np.argmin(self.W[i, nb])]
        if rng.random() < 0.5 * (P[i] + P[j]):
            return
        self.c[i] *= (1.0 - p['loss'])
        self.c[j] *= (1.0 - p['loss'])
        cand = np.flatnonzero(~self.A[i])
        cand = cand[cand != i]
        if cand.size == 0:
            return
        z = self.beta[i] * self.shat()[cand]
        wgt = np.exp(z - z.max())
        new = rng.choice(cand, p=wgt / wgt.sum())
        w = self.win()
        self.loss[w, deg[j]] += 1; self.gain[w, deg[new]] += 1
        self.A[i, j] = self.A[j, i] = False
        self.W[i, j] = self.W[j, i] = 0.0
        self.A[i, new] = self.A[new, i] = True
        self.W[i, new] = self.W[new, i] = p['w0']

plain, logged = ElementModel(seed=0), Logged(seed=0)
for _ in range(T_RUN):
    plain.step(); logged.step()
m1, m2 = plain.measures(), logged.measures()
print('check, the logged run follows the plain model step for step:',
      all(np.isclose(m1[key], m2[key]) for key in m1),
      f'(seed 0, mean direction preference {m1["beta_mean"]:.2f} in both)')

check, the logged run follows the plain model step for step: True (seed 0, mean direction preference 2.91 in both)


## Step 4. Thirty runs of the full model

In [7]:
RUNS = []
for sd in seeds(30):
    m = Logged(seed=sd)
    for _ in range(T_RUN):
        m.step()
    RUNS.append(dict(meet=m.meet, cp_str=m.cp_str, cp_cap=m.cp_cap, expo=m.expo, gain=m.gain,
                     loss=m.loss, traj=np.array(m.traj), kappa=m.measures()['kappa'],
                     beta=m.beta.mean()))
print(f'{len(RUNS)} runs; median degree heterogeneity at the end {np.median([r["kappa"] for r in RUNS]):.2f}, '
      f'median mean direction preference {np.median([r["beta"] for r in RUNS]):.2f}')

def share_hi(key):
    X = np.array([r[key] for r in RUNS])          # runs x windows x (lower, higher)
    return X[:, :, 1] / np.maximum(X.sum(axis=2), 1)

MEET, CPS, CPC = share_hi('meet'), share_hi('cp_str'), share_hi('cp_cap')
for tag, X in [('partner met along a coupling has the higher degree', MEET),
               ('copied partner is the stronger element', CPS),
               ('copied partner is the better paid', CPC)]:
    print(f'{tag}: median share {np.median(X[:, 0]):.3f} in the first 250 steps, '
          f'{np.median(X[:, -1]):.3f} in the last 250')

EARLY, LATE = slice(0, NWIN // 3), slice(2 * NWIN // 3, NWIN)
def rates(win):
    E = sum(r['expo'][win].sum(axis=0) for r in RUNS)
    G = sum(r['gain'][win].sum(axis=0) for r in RUNS)
    L = sum(r['loss'][win].sum(axis=0) for r in RUNS)
    return E, G, L
BINS = [2, 3, 4, 5, 6, 7, 8, 9, 11, 14, 18, 25, 35, 101]
def binned(E, X):
    kk = np.arange(KMAX); out = []
    for lo, hi in zip(BINS[:-1], BINS[1:]):
        sel = (kk >= lo) & (kk < hi)
        if E[sel].sum() > 2000:
            out.append(((kk[sel] * E[sel]).sum() / E[sel].sum(), X[sel].sum() / E[sel].sum()))
    return np.array(out)

N_EL, L_EL = 100, 300
RATE = {}
for tag, win in [('early', EARLY), ('late', LATE)]:
    E, G, L = rates(win)
    r_att = G.sum() / (E.sum() / N_EL)                 # attachments per step
    kk = np.arange(KMAX); ok = E > 2000
    y = G[ok] / E[ok] / r_att - kk[ok] / (2 * L_EL); x = 1 / N_EL - kk[ok] / (2 * L_EL)
    p_fit = (E[ok] * x * y).sum() / (E[ok] * x * x).sum()
    RATE[tag] = dict(gain=binned(E, G), loss=binned(E, L), r=r_att, p=p_fit)
    g, l = RATE[tag]['gain'], RATE[tag]['loss']
    print(f'{tag} practice: {r_att:.2f} attachments per step; random share p of the copying equation '
          f'fitted to the attachment rate = {p_fit:.2f}')
    for lo, hi in [(3, 6), (6, 13), (20, KMAX)]:
        sel = (kk >= lo) & (kk < hi)
        if E[sel].sum() > 2000:
            print(f'   degree {lo} to {hi - 1}: {1000 * G[sel].sum() / E[sel].sum():.1f} attachments and '
                  f'{1000 * L[sel].sum() / E[sel].sum():.1f} removals per element per 1,000 steps')

TR = np.array([r['traj'] for r in RUNS])          # runs x samples x (t, mean beta, mean s_hat)
Q = 1 / (1 + TR[:, :, 1] * TR[:, :, 2])
late = TR[0, :, 0] > 2 * T_RUN / 3
print(f'random share expected from the preference to first order, 1 / (1 + beta <s_hat>), '
      f'late practice: median {np.median(Q[:, late]):.2f}')

30 runs; median degree heterogeneity at the end 12.49, median mean direction preference 1.72
partner met along a coupling has the higher degree: median share 0.578 in the first 250 steps, 0.727 in the last 250
copied partner is the stronger element: median share 0.710 in the first 250 steps, 0.877 in the last 250
copied partner is the better paid: median share 0.811 in the first 250 steps, 0.943 in the last 250
early practice: 1.57 attachments per step; random share p of the copying equation fitted to the attachment rate = 1.00
   degree 3 to 5: 16.3 attachments and 12.0 removals per element per 1,000 steps
   degree 6 to 12: 15.1 attachments and 18.8 removals per element per 1,000 steps
late practice: 1.58 attachments per step; random share p of the copying equation fitted to the attachment rate = 0.89
   degree 3 to 5: 15.3 attachments and 14.5 removals per element per 1,000 steps
   degree 6 to 12: 16.3 attachments and 23.2 removals per element per 1,000 steps
   degree 20 to 100: 2

## Step 5. Figure S2

### The drawing style of the SI figures

In [8]:
import matplotlib as mpl
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.lines import Line2D

SI_RC = {'font.family': 'DejaVu Sans', 'font.size': 7, 'axes.labelsize': 6.6, 'xtick.labelsize': 6.0,
         'ytick.labelsize': 6.0, 'legend.fontsize': 6.0, 'axes.linewidth': 0.6, 'xtick.major.width': 0.6,
         'ytick.major.width': 0.6, 'xtick.major.size': 2.4, 'ytick.major.size': 0, 'legend.frameon': False,
         'axes.spines.top': True, 'axes.spines.right': True, 'mathtext.fontset': 'dejavusans'}
INK, NEUTRAL = '#1A1A1A', '#37474F'
BLUE, RED, GOLD, GREEN, GREY, NAVY = '#2E5090', '#A4161A', '#C9A227', '#2E7D4E', '#8A8A8A', '#1F3F7A'
SI_COND = {'baseline': '#C4C9CD', 'explore': '#8A8A8A', 'imitate': '#5F6B72', 'both': NAVY}
SI_LABEL = {'baseline': 'baseline', 'explore': 'exploration', 'imitate': 'imitation', 'both': 'both'}
_SI_HOT = plt.get_cmap('hot_r')(np.linspace(0.06, 0.94, 256))
_SI_COOL = plt.get_cmap('Blues')(np.linspace(0.92, 0.06, 256))
CMAP3 = LinearSegmentedColormap.from_list('hot_div', np.vstack([_SI_COOL, _SI_HOT]))
HOT3 = LinearSegmentedColormap.from_list('hot_half', CMAP3(np.linspace(0.50, 0.92, 256)))
SI_VLIM = 2.7157337518864986                       # the scale of Fig. 2 and of ref. 3, in bits
SI_CNORM = Normalize(-SI_VLIM, SI_VLIM)
DOT = (0, (1.2, 1.4))


def si_share_colour(w):
    # an element colored by its share against an even share, as in Figs. 2 and 3
    w = np.asarray(w, float); b = np.log2(w / w.sum() * len(w))
    return CMAP3(SI_CNORM(np.where(np.abs(b) < 1e-9, 0.0, b)))


def si_figure(w, h, label):
    plt.rcParams.update(SI_RC)
    fig = plt.figure(figsize=(w, h)); fig.set_label(label)
    return fig


def si_axes(fig, l, b, w, h):
    W, H = fig.get_size_inches()
    return fig.add_axes([l / W, b / H, w / W, h / H])


def si_head(fig, ax, letter, title, dy=0.12):
    # the title centered over the plot, the letter in bold beside its left end
    W, H = fig.get_size_inches()
    bb = ax.get_position()
    y = bb.y1 + dy / H
    t = fig.text((bb.x0 + bb.x1) / 2, y, title, fontsize=7.6, ha='center', va='bottom')
    ext = t.get_window_extent(fig.canvas.get_renderer()).transformed(fig.transFigure.inverted())
    fig.text(ext.x0 - 0.06 / W, y, letter, fontsize=9.0, fontweight='bold', ha='right', va='bottom')


def si_ticks(lo, hi):
    return [lo, (lo + hi) / 2, hi]


def si_nice(lo, hi, zero=False):
    # three round ticks, minimum, middle and maximum, as tight as round numbers allow around the data
    lo, hi = (min(0.0, lo), max(0.0, hi)) if zero else (lo, hi)
    span = (hi - lo) or abs(hi) or 1.0
    for e in range(int(np.floor(np.log10(span))) - 2, int(np.floor(np.log10(span))) + 2):
        for m in (1, 1.5, 2, 2.5, 3, 4, 5, 6, 8):
            u = m * 10.0 ** e
            a = np.floor(lo / u + 1e-9) * u
            b = np.ceil(hi / u - 1e-9) * u
            n = int(round((b - a) / u))
            if n % 2:
                b += u; n += 1
            if n <= 2:
                return [round(a, 10), round(a + u, 10), round(b, 10)]
    return [lo, (lo + hi) / 2, hi]


def si_frame(ax, xt=None, yt=None, xlog=False, ylog=False, xfmt=None, yfmt=None, grid=True):
    # a line above and below, none at the sides, ticks at the given values, a dotted rule at the middle tick
    if xlog:
        ax.set_xscale('log')
    if ylog:
        ax.set_yscale('log')
    g = lambda v, _: f'{v:,g}'
    if xt is not None:
        ax.set_xlim(xt[0], xt[-1]); ax.set_xticks(xt)
        ax.xaxis.set_major_formatter(mpl.ticker.FuncFormatter(xfmt or g))
        ax.xaxis.set_minor_locator(mpl.ticker.NullLocator())
    if yt is not None:
        ax.set_ylim(yt[0], yt[-1]); ax.set_yticks(yt)
        ax.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(yfmt or g))
        ax.yaxis.set_minor_locator(mpl.ticker.NullLocator())
        if grid:
            for v in yt[1:-1]:
                ax.axhline(v, color='gray', alpha=0.45, ls=':', lw=0.5, zorder=0)
    for s_ in ('left', 'right'):
        ax.spines[s_].set_visible(False)
    for s_ in ('top', 'bottom'):
        ax.spines[s_].set_visible(True); ax.spines[s_].set_linewidth(0.7)
    ax.tick_params(axis='y', length=0, pad=1.4)
    ax.tick_params(axis='x', length=2.4, width=0.6, direction='out', pad=1.4)
    ax.set_axisbelow(True)


def si_cbar(fig, l, b, w, h, cmap, lo, hi, label, fmt='{:g}'):
    ax = si_axes(fig, l, b, w, h)
    ax.imshow(np.linspace(1, 0, 128)[:, None], cmap=cmap, aspect='auto', extent=[0, 1, lo, hi])
    ax.set_xticks([]); ax.set_yticks([lo, hi]); ax.set_yticklabels([fmt.format(lo), fmt.format(hi)])
    ax.yaxis.tick_right(); ax.yaxis.set_label_position('right')
    for s_ in ax.spines.values():
        s_.set_visible(True); s_.set_linewidth(0.5); s_.set_linestyle(DOT)
    ax.tick_params(axis='y', length=1.6, pad=1.0, labelsize=5.6)
    ax.set_ylabel(label, fontsize=6.0, labelpad=1.5)
    return ax


def si_bars(ax, groups, colors, labels, dots=True):
    # bars at the median of each group with every run as a dot beside it
    for i, v in enumerate(groups):
        v = np.asarray(v, float)
        ax.bar(i, np.median(v), color=colors[i], width=0.58, zorder=2)
        if dots:
            ax.plot(np.full(len(v), i) + 0.20 + np.linspace(-0.05, 0.05, len(v)), v, '.', ms=1.8, color=INK,
                    alpha=0.6, zorder=3)
    ax.set_xticks(range(len(groups))); ax.set_xticklabels(labels)
    ax.set_xlim(-0.6, len(groups) - 0.4)


def si_bracket(ax, i, j, y, h, text):
    ax.plot([i, i, j, j], [y, y + h, y + h, y], '-', lw=0.6, color=INK)
    ax.text((i + j) / 2, y + h, text, ha='center', va='bottom', fontsize=6.4)


def si_radial(A, n_min=2, dmax=None):
    # elements at the angle of their place on the ring, drawn inward by degree, as in Fig. 5C
    d = A.sum(1).astype(float); n = len(d)
    th = np.pi / 2 - 2 * np.pi * np.arange(n) / n
    dmax = max(d.max() if dmax is None else dmax, n_min + 1)
    r = np.clip(np.log(dmax / np.maximum(d, n_min)) / np.log(dmax / n_min), 0, 1)
    return np.c_[r * np.cos(th), r * np.sin(th)], d


def si_structure(ax, A, W=None, isE=None, s=None, lim=1.12, dmax=None):
    # a learned structure: couplings darker and wider with weight, elements colored by their share of the
    # coupling against an even share and sized by degree, the task-essential elements ringed
    from matplotlib.collections import LineCollection
    xy, d = si_radial(A, dmax=dmax)
    iu = np.array(np.where(np.triu(A))).T
    if W is not None:
        s = (W * A).sum(1) if s is None else s
        ww = W[iu[:, 0], iu[:, 1]]; x = np.sqrt((ww - ww.min()) / max(ww.max() - ww.min(), 1e-12))
    else:
        x = np.full(len(iu), 0.3)
    o = np.argsort(x, kind='stable')
    ax.add_collection(LineCollection(xy[iu[o]], colors=plt.get_cmap('Greys')(0.25 + 0.55 * x[o]),
                                     linewidths=0.12 + 0.8 * x[o], alpha=0.75, zorder=1))
    size = 3.0 + 110.0 * ((d - 1) / (len(d) - 2)) ** 1.15
    o = np.argsort(d, kind='stable')
    ax.scatter(xy[o, 0], xy[o, 1], s=size[o], c=si_share_colour(s)[o], edgecolors=NEUTRAL, linewidths=0.25, zorder=3)
    if isE is not None:
        e = np.where(isE)[0]
        ax.scatter(xy[e, 0], xy[e, 1], s=(np.sqrt(size[e]) + 2.6) ** 2, facecolors='none', edgecolors=INK,
                   linewidths=0.6, zorder=4)
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect('equal'); ax.axis('off')

In [9]:
fig = si_figure(6.93, 3.60, 'SI Fig S2')
PW, PH = 1.62, 0.92
XS = [0.42, 2.78, 5.14]
YS = [2.24, 0.45]

def panel(i, j):
    return si_axes(fig, XS[j], YS[i], PW, PH)

def logticks(lo, hi):
    a, b = int(np.floor(np.log10(lo))), int(np.ceil(np.log10(hi)))
    if (b - a) % 2:
        b += 1
    return [10.0 ** a, 10.0 ** ((a + b) // 2), 10.0 ** b]

C0, C25, C5 = NAVY, GREY, GREEN
ax = panel(0, 0)
kx = np.linspace(3, 30, 50)
for (p, var), col, mk, lab in [((0.0, 'link'), C0, 'o', 'copied link, $p$ = 0'),
                               ((0.5, 'link'), C5, 'o', 'copied link, $p$ = 0.5'),
                               ((0.0, 'partner'), RED, 's', 'partner of a random element')]:
    k, est = KER[(p, var)]; sel = k <= 30
    ax.plot(k[sel], est[sel], mk, ms=2.4, mfc='white' if var == 'partner' else col, mec=col, mew=0.7, label=lab)
for p, col in [(0.0, C0), (0.5, C5)]:
    ax.plot(kx, p * 2 * M_GROW + (1 - p) * kx, '-', color=col, lw=0.8)
top = max(KER[(0.0, 'partner')][1][KER[(0.0, 'partner')][0] <= 30].max(), 30)
si_frame(ax, [0, 15, 30], [0, 30, 60])
ax.legend(loc='upper left', handletextpad=0.2, labelspacing=0.2, borderaxespad=0.1, fontsize=5.4)
ax.set_xlabel(r'degree $k$', labelpad=1); ax.set_ylabel(r'$2L\,\Pi(k)$', labelpad=1)
si_head(fig, ax, 'A', 'Copying is preferential')

ax = panel(0, 1)
lo_d, hi_d = min(np.min(DEGS[p]) for p in DEGS), max(np.max(DEGS[p]) for p in DEGS)
for p, col in [(0.0, C0), (0.25, C25), (0.5, C5)]:
    d = np.sort(DEGS[p]); cc = 1 - np.arange(len(d)) / len(d)
    ax.plot(d, cc, '-', color=col, lw=1.0, label=f'{p:g}, {(3 - p) / (1 - p):.2f}')
yl = min(1 - (len(DEGS[p]) - 1) / len(DEGS[p]) for p in DEGS)
si_frame(ax, [3, 30, 300], [1e-4, 1e-2, 1], xlog=True, ylog=True,
         xfmt=lambda v, _: f'{v:g}', yfmt=lambda v, _: f'{v:g}')
ax.legend(loc='lower left', handlelength=1.2, labelspacing=0.15, borderaxespad=0.05, fontsize=5.4,
          title=r'$p$, exponent', title_fontsize=5.4, alignment='left')
ax.set_xlabel(r'degree $k$', labelpad=1); ax.set_ylabel(r'$P(\geq k)$', labelpad=1)
si_head(fig, ax, 'B', 'Degrees of the growth')

ax = panel(0, 2)
for s_, col, ls in [(0.0, GREY, '--'), (0.5, '#B7C0C7', '-'), (2.0, RED, '-'), (8.0, NAVY, '-')]:
    ax.plot(dc, fermi(dc, s_), ls, color=col, lw=1.0, label=f'$s$ = {s_:g}')
ax.text(dc.max(), 0.53, 'copying model', fontsize=5.6, ha='right', va='bottom', color=NEUTRAL)
xt = si_nice(dc.min(), dc.max())
si_frame(ax, xt, [0, 0.5, 1])
ax.legend(loc='upper left', handlelength=1.4, labelspacing=0.25)
ax.set_xlabel(r'capability of the partner less own, $\Delta c$', labelpad=1)
ax.set_ylabel('copy probability', labelpad=1)
si_head(fig, ax, 'C', 'The Fermi rule, Eq. 10')

ax = panel(1, 0)
tw = (np.arange(NWIN) + 0.5) * T_RUN / NWIN
for X, col, lab in [(MEET, GREY, 'partner met has the higher degree'),
                    (CPS, BLUE, 'partner copied is the stronger'),
                    (CPC, RED, 'partner copied is the better paid')]:
    q1, q2, q3 = np.percentile(X, [25, 50, 75], axis=0)
    ax.fill_between(tw, q1, q3, color=col, alpha=0.18, lw=0); ax.plot(tw, q2, '-', color=col, lw=1.0, label=lab)
si_frame(ax, [0, T_RUN // 2, T_RUN], [0, 0.5, 1])
ax.text(T_RUN, 0.47, 'chance', fontsize=5.6, ha='right', va='top', color=NEUTRAL)
ax.legend(loc='lower right', handlelength=1.4, labelspacing=0.25, borderaxespad=0.1)
ax.set_xlabel('step', labelpad=1); ax.set_ylabel('share of draws or copies', labelpad=1)
si_head(fig, ax, 'D', 'Whom an element meets and copies')

for col_i, key, ttl, letter in [(1, 'gain', 'Attachment of a coupling', 'E'), (2, 'loss', 'Removal of a coupling', 'F')]:
    ax = panel(1, col_i)
    for tag, col in [('early', GREY), ('late', NAVY)]:
        X = RATE[tag][key]
        ax.plot(X[:, 0], 1000 * X[:, 1], 'o-', ms=2.4, lw=0.9, color=col,
                label=f'{tag}' + (f', fitted $p$ = {RATE[tag]["p"]:.2f}' if key == 'gain' else ''))
    r = RATE['late']['r']; kx = np.geomspace(2, 64, 60)
    if key == 'gain':
        p_l = RATE['late']['p']
        ax.plot(kx, 1000 * r * (p_l / N_EL + (1 - p_l) * kx / (2 * L_EL)), '-', color=NAVY, lw=0.6, alpha=0.6)
        ax.plot(kx, 1000 * r / N_EL + 0 * kx, ':', color=INK, lw=0.7)
        ax.text(64, 1000 * r / N_EL - 1.5, 'random, $p$ = 1', fontsize=5.6, ha='right', va='top')
        kl = kx[1000 * r * kx / (2 * L_EL) <= 32]
        ax.plot(kl, 1000 * r * kl / (2 * L_EL), '--', color=INK, lw=0.7)
        ax.text(kl[-1] * 1.05, 32, 'linear, $p$ = 0', fontsize=5.6, ha='left', va='center')
        ax.set_ylabel('attachments per 1,000 steps', labelpad=1)
    else:
        G = RATE['late']['gain']
        ax.plot(G[:, 0], 1000 * G[:, 1], '--', color=NAVY, lw=0.6, alpha=0.6, label='late attachment, from E')
        ax.set_ylabel('removals per 1,000 steps', labelpad=1)
    si_frame(ax, [2, 8, 32 * 2], [0, 30, 60], xlog=True, xfmt=lambda v, _: f'{v:g}')
    ax.set_xticks([2, 8, 64]); ax.set_xlim(1.8, 70)
    ax.legend(loc='upper left', handlelength=1.4, labelspacing=0.25)
    ax.set_xlabel(r'degree $k$ of the element', labelpad=1)
    si_head(fig, ax, letter, ttl)
plt.show()